# 02 — Data Cleaning & Staging

## Purpose
This notebook converts the profiled raw Online Retail II dataset into cleaned staging datasets for downstream analytics.

## Objectives
This notebook will:
- load the raw Excel workbook
- standardize column names
- remove exact duplicates
- separate returns/cancellations
- separate accounting adjustments
- create the cleaned main sales staging dataset
- generate a formal cleaning audit log
- generate a non-merchandise stock code reference list

## Important boundary
This notebook creates **staging outputs**, not final analytical outputs.

Permanent staging outputs from this notebook should be:
- `sales_main.csv`
- `returns_cancellations.csv`
- `accounting_adjustments.csv`

Derived datasets such as RFM base should be rebuilt later from PostgreSQL, not stored here as permanent source-like files.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
file_path = project_root / "data" / "raw" / "online_retail_ii" / "online_retail_II.xlsx"

print("Project root:", project_root)
print("File path:", file_path)
print("File exists:", file_path.exists())

Project root: /Users/pratikchetry/Desktop/retail-revenue-intelligence
File path: /Users/pratikchetry/Desktop/retail-revenue-intelligence/data/raw/online_retail_ii/online_retail_II.xlsx
File exists: True


In [11]:
sheet_1 = "Year 2009-2010"
sheet_2 = "Year 2010-2011"

df_2009_2010 = pd.read_excel(file_path, sheet_name=sheet_1)
df_2010_2011 = pd.read_excel(file_path, sheet_name=sheet_2)

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)
print("Combined raw shape:", df.shape)

df.head()

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)
Combined raw shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Step 1 — Standardize columns

Column names are standardized so later cleaning and loading logic uses one consistent naming convention.

In [12]:
df_raw = df.copy()
df_work = df_raw.copy()

df_work.columns = (
    df_work.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Standardized columns:")
print(df_work.columns.tolist())

df_work.head()

Standardized columns:
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Step 2 — Remove exact duplicate rows

Exact duplicate rows must be removed before any revenue or quantity logic is trusted.

This step is also recorded in the cleaning audit log.

In [13]:
audit = []

before = len(df_work)
duplicate_count_before = int(df_work.duplicated().sum())

df_work = df_work.drop_duplicates().copy()

after = len(df_work)

audit.append({
    "step": "remove_exact_duplicates",
    "rows_before": before,
    "rows_after": after,
    "rows_removed": before - after,
    "reason": "Exact duplicate rows across all raw columns"
})

print("Duplicate rows before removal:", duplicate_count_before)
print("Shape before duplicate removal:", before)
print("Shape after duplicate removal:", after)

pd.DataFrame(audit)

Duplicate rows before removal: 34335
Shape before duplicate removal: 1067371
Shape after duplicate removal: 1033036


,step,rows_before,rows_after,rows_removed,reason
0,remove_exact_duplicates,1067371,1033036,34335,Exact duplicate rows across all raw columns


## Stage 1 result — raw structure and duplicate removal

The two yearly sheets were combined successfully into one raw transaction dataset with **1,067,371 rows** and **8 columns**.

After standardizing column names, **34,335 exact duplicate rows** were removed, reducing the working dataset to **1,033,036 rows**.

### Business implication
If duplicates had not been removed, revenue and quantity metrics would have been overstated.

### Design implication
All downstream cleaning, PostgreSQL loading, and dashboard logic must use the de-duplicated working dataset, not the raw combined dataset.

## Stage 2 — Separate returns/cancellations and accounting adjustments

In this stage, we will identify and separate:
- return/cancellation rows
- accounting adjustment rows
- broad positive-sales rows

This is necessary because these record types cannot be mixed into one sales table without making revenue analysis misleading.

In [14]:
df_work["is_negative_quantity"] = df_work["quantity"] < 0
df_work["is_negative_price"] = df_work["price"] < 0
df_work["invoice_str"] = df_work["invoice"].astype(str)
df_work["is_return_invoice"] = df_work["invoice_str"].str.startswith("C", na=False)
df_work["is_adjustment_row"] = df_work["price"] < 0

df_work[[
    "invoice", "quantity", "price",
    "is_negative_quantity", "is_negative_price",
    "is_return_invoice", "is_adjustment_row"
]].head()

,invoice,quantity,price,is_negative_quantity,is_negative_price,is_return_invoice,is_adjustment_row
0,489434,12,6.95,False,False,False,False
1,489434,12,6.75,False,False,False,False
2,489434,12,6.75,False,False,False,False
3,489434,48,2.10,False,False,False,False
4,489434,24,1.25,False,False,False,False


In [15]:
df_adjustments = df_work[df_work["is_adjustment_row"]].copy()
df_returns = df_work[df_work["is_negative_quantity"] | df_work["is_return_invoice"]].copy()

audit.append({
    "step": "separate_accounting_adjustments",
    "rows_before": len(df_work),
    "rows_after": len(df_work) - len(df_adjustments),
    "rows_removed": len(df_adjustments),
    "reason": "Negative price rows treated as accounting adjustments"
})

audit.append({
    "step": "separate_returns_cancellations",
    "rows_before": len(df_work),
    "rows_after": len(df_work) - len(df_returns),
    "rows_removed": len(df_returns),
    "reason": "Negative quantity or C-prefixed invoice rows treated as returns/cancellations"
})

print("Adjustment rows shape:", df_adjustments.shape)
print("Return/cancellation rows shape:", df_returns.shape)

pd.DataFrame(audit)

Adjustment rows shape: (5, 13)
Return/cancellation rows shape: (22497, 13)


,step,rows_before,rows_after,rows_removed,reason
0,remove_exact_duplicates,1067371,1033036,34335,Exact duplicate rows across all raw columns
1,separate_accounting_adjustments,1033036,1033031,5,Negative price rows treated as accounting adju...
2,separate_returns_cancellations,1033036,1010539,22497,Negative quantity or C-prefixed invoice rows t...


In [16]:
df_positive_sales = df_work[
    (df_work["quantity"] > 0) &
    (df_work["price"] >= 0)
].copy()

print("Positive sales shape:", df_positive_sales.shape)
df_positive_sales.head()

Positive sales shape: (1010535, 13)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_negative_quantity,is_negative_price,invoice_str,is_return_invoice,is_adjustment_row
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,489434,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,489434,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,489434,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,489434,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,489434,False,False


## Stage 2 result — returns, adjustments, and positive-sales split

After duplicate removal, the working dataset contained **1,033,036 rows**.

From this cleaned working base:
- **5 rows** were identified as accounting adjustments
- **22,497 rows** were identified as returns/cancellations
- **1,010,535 rows** remained in the broad positive-sales working dataset

### Business implication
This confirms that the raw retail file contains multiple transaction behaviors, not just standard sales. Mixing returns, adjustments, and positive sales into one reporting table would produce misleading revenue metrics.

### Design implication
The project must maintain separate treatment for:
- standard positive sales
- returns/cancellations
- accounting adjustments

The broad positive-sales dataset is still not final, because it may still contain missing customer IDs, missing descriptions, zero-price rows, and non-merchandise codes.

## Stage 3 — Inspect remaining issues inside positive sales

The broad positive-sales dataset still needs review before it can become the final staging sales table.

In this stage, we will inspect:
- missing customer IDs
- missing descriptions
- zero-price rows

These checks will determine what belongs in the final `sales_main.csv`.

In [17]:
print("Missing values in positive sales dataset:")
display(df_positive_sales.isna().sum().sort_values(ascending=False))

missing_customer_positive = int(df_positive_sales["customer_id"].isna().sum())
missing_description_positive = int(df_positive_sales["description"].isna().sum())
zero_price_rows = int((df_positive_sales["price"] == 0).sum())

print("Missing customer_id in positive sales:", missing_customer_positive)
print("Missing description in positive sales:", missing_description_positive)
print("Rows with zero price in positive sales:", zero_price_rows)

Missing values in positive sales dataset:


customer_id             231040
description               1642
invoice                      0
stockcode                    0
quantity                     0
invoicedate                  0
price                        0
country                      0
is_negative_quantity         0
is_negative_price            0
invoice_str                  0
is_return_invoice            0
is_adjustment_row            0
dtype: int64

Missing customer_id in positive sales: 231040
Missing description in positive sales: 1642
Rows with zero price in positive sales: 2621


In [18]:
display(df_positive_sales[df_positive_sales["price"] == 0].head(10))
display(df_positive_sales[df_positive_sales["customer_id"].isna()].head(10))
display(df_positive_sales[df_positive_sales["description"].isna()].head(10))

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_negative_quantity,is_negative_price,invoice_str,is_return_invoice,is_adjustment_row
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,False,False,489659,False,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,False,False,489781,False,False
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,False,False,489825,False,False
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom,False,False,489861,False,False
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,False,False,489882,False,False
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom,False,False,489898,False,False
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom,False,False,489903,False,False
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,False,False,489998,False,False
7204,490015,21982,NaN,467,2009-12-03 12:29:00,0.0,NaN,United Kingdom,False,False,490015,False,False
9249,490123,84508B,NaN,184,2009-12-03 18:08:00,0.0,NaN,United Kingdom,False,False,490123,False,False


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_negative_quantity,is_negative_price,invoice_str,is_return_invoice,is_adjustment_row
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,False,False,489525,False,False
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,False,False,489525,False,False
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False,False,489548,False,False
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,False,False,489548,False,False
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False,False,489548,False,False
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,False,False,489548,False,False
1059,489548,22131,FOOD CONTAINER SET 3 LOVE HEART,2,2009-12-01 12:32:00,1.95,NaN,United Kingdom,False,False,489548,False,False
1060,489548,22079,RIBBON REEL HEARTS DESIGN,10,2009-12-01 12:32:00,1.65,NaN,United Kingdom,False,False,489548,False,False
1061,489548,22138,BAKING SET 9 PIECE RETROSPOT,3,2009-12-01 12:32:00,4.95,NaN,United Kingdom,False,False,489548,False,False
1062,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,NaN,United Kingdom,False,False,489548,False,False


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_negative_quantity,is_negative_price,invoice_str,is_return_invoice,is_adjustment_row
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,False,False,489659,False,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,False,False,489781,False,False
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,False,False,489882,False,False
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom,False,False,489898,False,False
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom,False,False,489903,False,False
7204,490015,21982,NaN,467,2009-12-03 12:29:00,0.0,NaN,United Kingdom,False,False,490015,False,False
9249,490123,84508B,NaN,184,2009-12-03 18:08:00,0.0,NaN,United Kingdom,False,False,490123,False,False
10236,490150,84347,NaN,80,2009-12-04 09:44:00,0.0,NaN,United Kingdom,False,False,490150,False,False
14503,490543,37446,NaN,222,2009-12-07 09:45:00,0.0,NaN,United Kingdom,False,False,490543,False,False
15489,490688,21133,NaN,270,2009-12-07 13:50:00,0.0,NaN,United Kingdom,False,False,490688,False,False


## Stage 3 result — remaining issues inside positive sales

The broad positive-sales dataset contains **1,010,535 rows**, but it is still not ready to become the final staging sales table.

### Confirmed remaining issues
- Missing `customer_id`: **231,040**
- Missing `description`: **1,642**
- Zero-price rows: **2,621**

### Business implication
This confirms that not every positive-quantity row represents a valid sellable sales record for downstream reporting. In particular:
- rows with missing `description` weaken product-level reporting
- zero-price rows may represent gifts, samples, service entries, or other non-standard transactions
- missing `customer_id` rows may still belong in overall sales analysis, but they cannot support customer-level analytics directly

### Design implication
The final staging sales dataset must exclude:
- zero-price rows
- rows with missing `description`

Rows with missing `customer_id` can remain in `sales_main.csv`, but customer-level analytics will later be rebuilt separately from PostgreSQL using only customer-linked transactions.

## Stage 4 — Create the final main sales staging dataset

In this stage, we create the permanent main staging sales dataset.

Rules:
- `quantity > 0`
- `price > 0`
- `description` is not missing

Rows with missing `customer_id` may still remain here, because total sales reporting is broader than customer-level analytics.

In [19]:
df_sales_main = df_positive_sales[
    (df_positive_sales["price"] > 0) &
    (df_positive_sales["description"].notna())
].copy()

df_sales_main["revenue"] = df_sales_main["quantity"] * df_sales_main["price"]

audit.append({
    "step": "create_sales_main",
    "rows_before": len(df_positive_sales),
    "rows_after": len(df_sales_main),
    "rows_removed": len(df_positive_sales) - len(df_sales_main),
    "reason": "Removed zero-price rows and rows with missing description from positive sales"
})

print("Main sales dataset shape:", df_sales_main.shape)
print("Rows removed from positive sales to create sales_main:", len(df_positive_sales) - len(df_sales_main))

display(df_sales_main["revenue"].describe())
df_sales_main.head()

Main sales dataset shape: (1007914, 14)
Rows removed from positive sales to create sales_main: 2621


count    1.007914e+06
mean     2.031585e+01
std      2.057160e+02
min      1.000000e-03
25%      4.130000e+00
50%      1.008000e+01
75%      1.770000e+01
max      1.684696e+05
Name: revenue, dtype: float64

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_negative_quantity,is_negative_price,invoice_str,is_return_invoice,is_adjustment_row,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,489434,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,489434,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,489434,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,489434,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,489434,False,False,30.0


In [20]:
print("Missing values in sales_main:")
display(df_sales_main.isna().sum().sort_values(ascending=False))

Missing values in sales_main:


customer_id             228489
invoice                      0
stockcode                    0
description                  0
quantity                     0
invoicedate                  0
price                        0
country                      0
is_negative_quantity         0
is_negative_price            0
invoice_str                  0
is_return_invoice            0
is_adjustment_row            0
revenue                      0
dtype: int64

## Stage 4 result — final main sales staging dataset

The final main sales staging dataset contains **1,007,914 rows**.

To create this table, **2,621 rows** were removed from the broad positive-sales dataset.

### What was removed
These removed rows were the remaining:
- zero-price rows
- rows with missing `description`

### Revenue baseline
The resulting staging sales dataset has a total revenue baseline of approximately **20.48M**, with a median line-level revenue of **10.08** and a highly right-skewed revenue distribution.

### Customer ID status
`customer_id` is still missing in **228,489** rows.

### Business implication
This confirms that the final staging sales dataset is suitable for:
- total revenue analysis
- product analysis
- country analysis
- time-series analysis

But it is **not automatically suitable for customer-level analysis**, because a substantial share of sales rows still do not have a customer identifier.

### Design implication
Customer segmentation and RFM logic must later be rebuilt from PostgreSQL using only customer-linked transactions, rather than being treated as a peer source file in the staging layer.

## Stage 5 — Build the initial non-merchandise reference list

Some stock codes represent operational or service-like entries rather than real merchandise.

This stage builds an initial reference list for those codes so later:
- product dashboards can filter them correctly
- PostgreSQL can store merchandise flags
- revenue views can distinguish goods from service-style entries

In [21]:
product_revenue_summary = (
    df_sales_main.groupby(["stockcode", "description"], as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        total_quantity=("quantity", "sum"),
        total_orders=("invoice", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

display(product_revenue_summary.head(30))

,stockcode,description,total_revenue,total_quantity,total_orders
5615,M,Manual,339599.81,9630,785
1757,22423,REGENCY CAKESTAND 3 TIER,330590.32,26478,3918
5614,DOT,DOTCOM POSTAGE,309854.11,1415,1415
5157,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257546.20,94142,5356
3221,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
3372,47566,PARTY BUNTING,148318.28,28200,2674
5139,85099B,JUMBO BAG RED RETROSPOT,145961.83,77280,3245
3716,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807
5617,POST,POSTAGE,125682.42,5363,1851
1343,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018


In [22]:
known_non_merchandise_codes = ["M", "DOT", "POST"]

non_merchandise_codes = product_revenue_summary[
    product_revenue_summary["stockcode"].isin(known_non_merchandise_codes)
].copy()

non_merchandise_codes["product_type"] = non_merchandise_codes["stockcode"].map({
    "M": "service/manual",
    "DOT": "postage/service",
    "POST": "postage/service"
})

non_merchandise_codes["is_merchandise"] = False

display(non_merchandise_codes)

,stockcode,description,total_revenue,total_quantity,total_orders,product_type,is_merchandise
5615,M,Manual,339599.81,9630,785,service/manual,False
5614,DOT,DOTCOM POSTAGE,309854.11,1415,1415,postage/service,False
5617,POST,POSTAGE,125682.42,5363,1851,postage/service,False


## Stage 5 result — initial non-merchandise reference list

The product revenue ranking confirms that some high-revenue stock codes are **not merchandise**.

### Confirmed non-merchandise codes
- `M` → **Manual** → **339,599.81**
- `DOT` → **DOTCOM POSTAGE** → **309,854.11**
- `POST` → **POSTAGE** → **125,682.42**

### Business implication
If these codes are left inside normal product reporting, top-product charts and product revenue rankings will be misleading. A dashboard could incorrectly present postage or manual entries as if they were sellable products.

### Design implication
The project must carry forward an explicit non-merchandise reference list.  
Later in PostgreSQL, the product dimension must support fields such as:
- `is_merchandise`
- `product_type`

All product dashboards and product SQL views must be able to filter merchandise vs non-merchandise explicitly.


## Stage 6 — Save permanent staging outputs and audit log

In this stage, we save only the permanent staging outputs from cleaning.

Permanent staging outputs:
- `sales_main.csv`
- `returns_cancellations.csv`
- `accounting_adjustments.csv`
- `non_merchandise_codes.csv`

We also save:
- `cleaning_audit_log.csv`

Helper columns used during cleaning are removed before saving the staging files.

In [23]:
staging_path = project_root / "data" / "staging"
reports_path = project_root / "outputs" / "reports"

staging_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)

print("Staging path:", staging_path)
print("Reports path:", reports_path)

Staging path: /Users/pratikchetry/Desktop/retail-revenue-intelligence/data/staging
Reports path: /Users/pratikchetry/Desktop/retail-revenue-intelligence/outputs/reports


In [24]:
sales_main_save = df_sales_main[
    ["invoice", "stockcode", "description", "quantity", "invoicedate", "price", "customer_id", "country", "revenue"]
].copy()

returns_save = df_returns[
    ["invoice", "stockcode", "description", "quantity", "invoicedate", "price", "customer_id", "country"]
].copy()

adjustments_save = df_adjustments[
    ["invoice", "stockcode", "description", "quantity", "invoicedate", "price", "customer_id", "country"]
].copy()

sales_main_save.to_csv(staging_path / "sales_main.csv", index=False)
returns_save.to_csv(staging_path / "returns_cancellations.csv", index=False)
adjustments_save.to_csv(staging_path / "accounting_adjustments.csv", index=False)
non_merchandise_codes.to_csv(staging_path / "non_merchandise_codes.csv", index=False)

pd.DataFrame(audit).to_csv(reports_path / "cleaning_audit_log.csv", index=False)

print("Staging datasets and audit log saved successfully.")

Staging datasets and audit log saved successfully.


In [25]:
print("Saved staging files:")
for f in staging_path.glob("*.csv"):
    print("-", f.name)

print("\nSaved report files:")
for f in reports_path.glob("*.csv"):
    print("-", f.name)

Saved staging files:
- returns_cancellations.csv
- non_merchandise_codes.csv
- sales_main.csv
- accounting_adjustments.csv

Saved report files:
- cleaning_audit_log.csv


In [26]:
audit_df = pd.read_csv(reports_path / "cleaning_audit_log.csv")
audit_df

,step,rows_before,rows_after,rows_removed,reason
0,remove_exact_duplicates,1067371,1033036,34335,Exact duplicate rows across all raw columns
1,separate_accounting_adjustments,1033036,1033031,5,Negative price rows treated as accounting adju...
2,separate_returns_cancellations,1033036,1010539,22497,Negative quantity or C-prefixed invoice rows t...
3,create_sales_main,1010535,1007914,2621,Removed zero-price rows and rows with missing ...


## Stage 6 result — permanent staging outputs and audit log

The cleaning notebook has now produced the permanent staging layer for the project.

### Saved staging outputs
- `sales_main.csv`
- `returns_cancellations.csv`
- `accounting_adjustments.csv`
- `non_merchandise_codes.csv`

### Saved audit output
- `cleaning_audit_log.csv`

### Audit trail summary
The cleaning process recorded the following transformations:

- **34,335** exact duplicate rows removed
- **5** accounting adjustment rows separated
- **22,497** return/cancellation rows separated
- **2,621** remaining zero-price or missing-description rows removed from broad positive sales to create `sales_main`

### Business implication
The project now has a traceable staging layer and a formal audit trail. This makes the cleaning process explainable, reproducible, and reviewable.

### Design implication
From this point onward:
- PostgreSQL loading should use the **staging layer**
- `db_ready/` should not be treated as a permanent architecture layer
- derived analytics such as RFM should be rebuilt from PostgreSQL, not stored beside staging sources